In [ ]:
%load_ext autoreload

%autoreload 2

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

# ABO Furniture Dataset — Quality Check

Validates the downloaded ABO furniture subset: corrupt file detection, class balance, visual sample grids per class, and retrieval group spot-checks.

## Imports

In [ ]:
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import pyrootutils
from PIL import Image

## Parameters

In [ ]:
ROOT = pyrootutils.setup_root(
    search_from=".",
    indicator="pyproject.toml",
    project_root_env_var=True,
    dotenv=True,
    pythonpath=True,
    cwd=True,
)

SCRIPT_DIR = ROOT / "scripts" / "prepare_datasets" / "abo"
DATA_DIR = ROOT / "data" / "abo"
MANIFEST_PATH = SCRIPT_DIR / "abo_manifest.json"
GROUPS_PATH = SCRIPT_DIR / "abo_retrieval_groups.json"
SPLITS = ["train", "val", "test"]
CLASSES = ["bed", "chair", "lamp", "sofa", "storage", "table"]

random.seed(42)

print(f"ROOT: {ROOT}")
print(f"DATA_DIR: {DATA_DIR}")

## 1 — Corrupt File Check

PIL-open every JPEG under `data/abo/`. Any file that raises an exception is corrupt.

In [ ]:
all_jpgs = sorted(DATA_DIR.glob("*/*/*.jpg"))
print(f"Total JPEG files on disk: {len(all_jpgs)}")

corrupt = []
for path in all_jpgs:
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception as exc:
        corrupt.append((path, str(exc)))

if corrupt:
    print(f"\nCORRUPT FILES ({len(corrupt)}):")
    for p, err in corrupt:
        print(f"  {p.relative_to(ROOT)}  —  {err}")
else:
    print("No corrupt files found.")

assert len(corrupt) == 0, f"{len(corrupt)} corrupt file(s) found"
print("OK")

## 2 — Class Balance

Count images per class per split from the manifest. The ABO dataset is naturally imbalanced (chair >> bed) — we take all items without capping, so high ratios are expected and pedagogically useful for the retrieval exercise.

In [ ]:
import json

with MANIFEST_PATH.open() as f:
    manifest = json.load(f)

print(f"Manifest entries: {len(manifest)}")
print()

# Count items and images per split/class
img_counts = defaultdict(Counter)  # split -> class -> image count
item_counts = defaultdict(Counter)  # split -> class -> item count
item_sets = defaultdict(lambda: defaultdict(set))

for e in manifest:
    img_counts[e["split"]][e["label"]] += 1
    item_sets[e["split"]][e["label"]].add(e["item_id"])

for split in SPLITS:
    for cls in CLASSES:
        item_counts[split][cls] = len(item_sets[split][cls])

# Print table
col_w = 10
print(f"{'':12}" + "".join(f"{s:>{col_w}}" for s in SPLITS) + f"{'TOTAL':>{col_w}}")
print("-" * (12 + col_w * 4))
print("IMAGES")
for cls in CLASSES:
    row = f"  {cls:<10}"
    total = 0
    for split in SPLITS:
        n = img_counts[split][cls]
        row += f"{n:>{col_w}}"
        total += n
    row += f"{total:>{col_w}}"
    print(row)
print("ITEMS")
for cls in CLASSES:
    row = f"  {cls:<10}"
    total = 0
    for split in SPLITS:
        n = item_counts[split][cls]
        row += f"{n:>{col_w}}"
        total += n
    row += f"{total:>{col_w}}"
    print(row)

# Report ratio — no assertion, imbalance is inherent to ABO
print()
print("Class balance (max/min image ratio per split):")
for split in SPLITS:
    counts = [img_counts[split][cls] for cls in CLASSES]
    ratio = max(counts) / min(counts)
    dominant = max(CLASSES, key=lambda c: img_counts[split][c])
    smallest = min(CLASSES, key=lambda c: img_counts[split][c])
    print(
        f"  {split:<8} max={max(counts)} ({dominant}) min={min(counts)} ({smallest}) ratio={ratio:.1f}x"
    )
print("  (Expected — chair is the most common ABO category, bed the rarest.)")

## 3 — Visual Sample Grid

4×4 random training images per class. Visual check for blank images, wrong labels, and non-product content.

In [ ]:
N_COLS = 4
N_ROWS = 4

for cls in CLASSES:
    cls_dir = DATA_DIR / "train" / cls
    all_imgs = list(cls_dir.glob("*.jpg"))
    sample = random.sample(all_imgs, min(N_COLS * N_ROWS, len(all_imgs)))

    fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(N_COLS * 2.2, N_ROWS * 2.2))
    fig.suptitle(f"Class: {cls}  ({len(all_imgs)} train images)", fontsize=13, y=1.01)

    for ax, img_path in zip(axes.flat, sample, strict=False):
        ax.imshow(Image.open(img_path))
        ax.axis("off")
    for ax in axes.flat[len(sample) :]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## 4 — Retrieval Group Spot-Check

For 5 random groups with ≥2 images, display all views side-by-side to confirm they show the same product.

In [ ]:
import json

with GROUPS_PATH.open() as f:
    retrieval_groups = json.load(f)

print(f"Total retrieval groups: {len(retrieval_groups)}")

# Build image_id -> (split, label) lookup from manifest
img_lookup = {e["image_id"]: (e["split"], e["label"]) for e in manifest}

# Filter to groups with ≥2 images that are in the manifest
multi_view = [
    (item_id, g)
    for item_id, g in retrieval_groups.items()
    if len(g["image_ids"]) >= 2 and all(iid in img_lookup for iid in g["image_ids"])
]
print(f"Groups with ≥2 images in manifest: {len(multi_view)}")

spot_check = random.sample(multi_view, min(5, len(multi_view)))

for item_id, group in spot_check:
    img_ids = group["image_ids"]
    n = len(img_ids)
    fig, axes = plt.subplots(1, n, figsize=(n * 2.5, 2.8))
    if n == 1:
        axes = [axes]
    fig.suptitle(
        f"item_id={item_id}  label={group['label']}  split={group['split']}  ({n} views)",
        fontsize=10,
    )
    for ax, img_id in zip(axes, img_ids, strict=False):
        split, label = img_lookup[img_id]
        img_path = DATA_DIR / split / label / f"{img_id}.jpg"
        ax.imshow(Image.open(img_path))
        ax.set_title(f"view {img_ids.index(img_id)}", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 5 — Final Summary

Counts from both the manifest (items) and disk (images), side by side.

In [ ]:
# Items from manifest
manifest_items = defaultdict(lambda: defaultdict(set))
for e in manifest:
    manifest_items[e["split"]][e["label"]].add(e["item_id"])

# Images from disk
disk_counts = defaultdict(Counter)
for path in DATA_DIR.glob("*/*/*.jpg"):
    split, label = path.parts[-3], path.parts[-2]
    disk_counts[split][label] += 1

print("=" * 70)
print("FINAL DATASET SUMMARY")
print("=" * 70)

for split in SPLITS:
    n_items = sum(len(manifest_items[split][cls]) for cls in CLASSES)
    n_imgs = sum(disk_counts[split][cls] for cls in CLASSES)
    print(f"\n  [{split.upper()}]  {n_items} items  /  {n_imgs} images")
    print(f"  {'class':<12} {'items':>8} {'images':>8} {'img/item':>10}")
    print("  " + "-" * 40)
    for cls in CLASSES:
        ni = len(manifest_items[split][cls])
        nd = disk_counts[split][cls]
        ratio = nd / ni if ni > 0 else 0
        print(f"  {cls:<12} {ni:>8} {nd:>8} {ratio:>9.1f}")

total_items = sum(len(manifest_items[s][c]) for s in SPLITS for c in CLASSES)
total_imgs = sum(disk_counts[s][c] for s in SPLITS for c in CLASSES)
print(f"\n  TOTAL: {total_items} items  /  {total_imgs} images")
print("=" * 70)